[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/evals/bedrock_tracing_and_evals_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# 使用 OpenInference 和 Phoenix 为 AWS Bedrock 客户端插桩

本教程使用 OpenInference 追踪对 AWS Bedrock 的模型调用。OpenInference Bedrock 追踪器为 Python `boto3` 库插桩，因此所有 `invoke_model` 调用都会自动生成可发送至 Phoenix 的追踪。

ℹ️ 本 Notebook 需要有效的 AWS 配置、AWS Bedrock 和 Anthropic `claude-v2` 模型的访问权限，以及用于 LLM 裁判评估的 OpenAI API 密钥。

## 1. 安装依赖并设置 OpenTelemetry 追踪器

首先安装依赖。

In [ ]:
%pip install arize-phoenix boto3 openinference-instrumentation-bedrock

导入库。

In [ ]:
import json

以下环境变量可用于连接 Arize Phoenix 云实例。可在 [Phoenix Cloud](https://app.phoenix.arize.com) 账号设置页获取 API 密钥和端点。

如果希望自行托管 Phoenix，请参阅[自托管说明](https://arize.com/docs/phoenix/deployment)。云版本与自托管版本的功能相同。

In [ ]:
import os
from getpass import getpass

import boto3
from openinference.instrumentation.bedrock import BedrockInstrumentor
from phoenix.otel import SimpleSpanProcessor, register

if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 收集器端点：")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint

if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥：")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

这里通过添加两个 SpanProcessor 来配置 OpenTelemetry 追踪器。第一个将 OpenInference 插桩接收的全部追踪打印到控制台。第二个将追踪导出到 Phoenix，以便收集和查看。

In [ ]:
tracer_provider = register(project_name="bedrock-tracing-and-evals-tutorial", auto_instrument=True)
tracer_provider.add_span_processor(
    SimpleSpanProcessor(endpoint=os.getenv("PHOENIX_COLLECTOR_ENDPOINT"))
)

In [ ]:
# 另一种方式：若尚未设置，则提示输入 AWS 凭证
# 取消以下代码的注释，以交互方式输入凭证

if not os.getenv("AWS_ACCESS_KEY_ID"):
    os.environ["AWS_ACCESS_KEY_ID"] = getpass("🔑 请输入 AWS Access Key ID：")
if not os.getenv("AWS_SECRET_ACCESS_KEY"):
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("🔑 请输入 AWS Secret Access Key：")
# if not os.getenv("AWS_DEFAULT_REGION"):
#     os.environ["AWS_DEFAULT_REGION"] = input("🌍 请输入 AWS 区域，默认为 us-west-2：") or "us-west-2"

## 2. 为 Bedrock 客户端插桩

现在创建一个 `boto3` 会话，初始化与 AWS 服务交互的配置环境。如果尚未配置 `boto3` 使用你的凭证，请参阅[官方文档](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/configuration.html)。如果已安装 AWS CLI，也可以在终端运行 `aws configure`。

In [ ]:
session = boto3.session.Session()

使用此会话配置创建的客户端目前尚未插桩。我们先创建一个用于对比。

In [ ]:
uninstrumented_client = session.client("bedrock-runtime", region_name="us-west-2")

现在使用 OpenInference 插桩器为 Bedrock 插桩。此调用之后创建的所有 Bedrock 客户端，在调用 `invoke_model` 时都会自动生成追踪。

In [ ]:
BedrockInstrumentor().instrument(skip_dep_check=True)
instrumented_client = session.client("bedrock-runtime", region_name="us-west-2")

## 3. 调用 LLM 并查看 OpenInference 追踪

使用 `uninstrumented_client` 调用 `invoke_model` 不会生成追踪，但会显示 LLM 的输出。

In [ ]:
prompt = b"""{"prompt": "Human: What is the 3rd month of the year in alphabetical order? Assistant:", "max_tokens_to_sample": 1024}"""
response = uninstrumented_client.invoke_model(modelId="anthropic.claude-v2:1", body=prompt)
response_body = json.loads(response.get("body").read())
print(response_body["completion"])

使用 `instrumented_client` 调用 LLM 会将追踪打印到控制台！将 `SpanProcessor` 配置为导出到其他 OpenTelemetry 收集器，即可收集和分析 OpenInference span，更好地理解 LLM 应用的行为。

In [ ]:
response = instrumented_client.invoke_model(modelId="anthropic.claude-v2:1", body=prompt)
response_body = json.loads(response.get("body").read())
print(response_body["completion"])

## 4. 收集全部追踪与数据

使用 `instrumented_client` 收集全部追踪；本例使用一组知识问答。

In [ ]:
trivia_questions = [
    "What is the only U.S. state that starts with two vowels?",
    "What is the 3rd month of the year in alphabetical order?",
    "What is the capital of Mongolia?",
    "How many minutes are there in a leap year?",
    "If a train leaves New York at 3 PM traveling west at 60 mph, and another leaves Chicago at 4 PM traveling east at 80 mph, at what time will they meet?",
    "Which element has the chemical symbol 'Fe'?",
    "What five-letter word becomes shorter when you add two letters to it?",
    "What country has won the most FIFA World Cups?",
    "If today is Wednesday, what day of the week will it be 100 days from now?",
    "A farmer has 17 sheep and all but 9 run away. How many does he have left?",
]

for i, question in enumerate(trivia_questions, start=1):
    prompt_str = f"""
{{
    "prompt": "Human: {question} Assistant:",
    "max_tokens_to_sample": 300
}}
"""
    response = instrumented_client.invoke_model(
        modelId="anthropic.claude-v2:1",
        body=prompt_str.encode("utf-8"),
        contentType="application/json",
        accept="application/json",
    )

    response_body = json.loads(response.get("body").read())
    print(f"Q{i}: {question}")
    print(f"A{i}: {response_body['completion'].strip()}\n{'-' * 60}")

## 5. 设置并运行评估

将追踪导入为 DataFrame 后，调整列以适配评估模板。

In [ ]:
qa_template = """给定一个问题和答案，你必须判断答案是否正确回答了问题。数据如下：
    [BEGIN DATA]
    ************
    [Question]: {Question}
    ************
    [Answer]: {Answer}
    [END DATA]
回复必须是单个词，"correct" 或 "incorrect"，
除此之外不能包含任何文字或字符。
"correct" 表示答案正确且完整地回答了问题。
"incorrect" 表示答案不正确，或只部分回答了问题。"""

In [ ]:
from phoenix.client import Client

spans_df = Client().spans.get_spans_dataframe(project_name="bedrock-tracing-and-evals-tutorial")
spans_df.head()

In [ ]:
eval_df = spans_df[["context.span_id", "attributes.input.value", "attributes.output.value"]].copy()
eval_df.set_index("context.span_id", inplace=True)
eval_df.head()

In [ ]:
evals_copy = eval_df.copy()
evals_copy["attributes.input.value"] = (
    evals_copy["attributes.input.value"]
    .str.replace(r"^Human: ", "", regex=True)
    .str.replace(r"Assistant:$", "", regex=True)
)

evals_copy = evals_copy.rename(
    columns={"attributes.input.value": "Question", "attributes.output.value": "Answer"}
)
evals_copy.head()

In [ ]:
from openinference.instrumentation import suppress_tracing
from phoenix.evals import (
    ClassificationEvaluator,
    async_evaluate_dataframe,
)
from phoenix.evals.llm import LLM

llm = LLM(provider="openai", model="gpt-4")

qa_correctness_eval = ClassificationEvaluator(
    name="qa_correctness",
    prompt_template=qa_template,
    llm=llm,
    choices={"correct": 1.0, "incorrect": 0.0},
)
with suppress_tracing():
    Q_and_A_classifications = await async_evaluate_dataframe(
        dataframe=evals_copy, evaluators=[qa_correctness_eval]
    )
Q_and_A_classifications.head()

## 6. 将追踪记录到 Phoenix

In [ ]:
from phoenix.evals.utils import to_annotation_dataframe

relevancy_eval_df = to_annotation_dataframe(dataframe=Q_and_A_classifications)

Client().spans.log_span_annotations_dataframe(
    dataframe=relevancy_eval_df,
    annotator_kind="LLM",
)

![展示包含全部追踪和评估的 Phoenix UI 的 GIF](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/bedrock_tracing_eval_medium.gif)

有关插桩集成 OpenInference 的更多信息，请参阅[文档](https://arize.com/docs/phoenix/references/openinference)。